# Качество данных: панель Santander

Задача-легенда: банк хочет понять, кто уходит из продуктов, когда именно и что держит клиента.
Данные - соревнование Kaggle [Santander Product Recommendation](https://www.kaggle.com/c/santander-product-recommendation/data):
месячные снимки по клиентам за январь 2015 - май 2016, 24 признака клиента и 24 флага продуктов.
Описание всех полей - в `data/raw/santander_data_dictionary.md`.

Прежде чем считать когорты, надо понять, можно ли этой панели доверять. Проверяем по порядку:
объём и уникальность ключа, ровно ли растёт панель по месяцам, где пропуски и мусор,
нет ли дырок в истории клиентов и кого нельзя путать с оттоком.

Таблица большая - 13,6 млн строк, поэтому всё считаем в DuckDB, а в pandas забираем только агрегаты.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.functions import PRODUCTS, build_panel, n_prod_expr

pd.set_option('display.width', 200)

RAW = PROJECT_ROOT / 'data' / 'raw' / 'train_ver2.csv'
PARQUET = PROJECT_ROOT / 'data' / 'processed' / 'train.parquet'

con = duckdb.connect(config={'threads': 6})

csv весит 2,3 ГБ, и читать его при каждом запросе бессмысленно - один раз переложим в parquet
(со сжатием zstd это ~40 секунд и примерно в 5 раз меньше на диске).

`sample_size=-1` заставляет DuckDB просканировать файл целиком при определении типов. Без этого
он смотрит только начало файла и ошибается на колонках, где мусор появляется ближе к концу.

In [ ]:
if not PARQUET.exists():
    con.execute(f"""COPY (SELECT * FROM read_csv('{RAW}', header=true, sample_size=-1))
                    TO '{PARQUET}' (FORMAT parquet, COMPRESSION zstd)""")

build_panel(con, PARQUET)  # создаёт view t и таблицы p (панель) и fs (профиль клиента)
con.execute('DESCRIBE t').fetchdf()

Сразу видно первую проблему: `age`, `antiguedad`, `ind_nuevo`, `indrel`, `ind_actividad_cliente`,
а из продуктов `ind_nomina_ult1` и `ind_nom_pens_ult1` приехали как VARCHAR. Числовая колонка,
которая читается строкой, - почти всегда признак мусора внутри. Разберёмся ниже.

## Объём и ключ

Панель должна быть устроена так: одна строка на клиента и месяц. Проверим, что дублей нет.

In [ ]:
con.execute("""
SELECT count(*) AS строк,
       count(DISTINCT ncodpers) AS клиентов,
       count(DISTINCT (ncodpers, fecha_dato)) AS уникальных_пар
FROM t""").fetchdf()

13 647 309 строк, 956 645 клиентов, и число уникальных пар (клиент, месяц) совпадает с числом
строк - дублей нет, ключ честный.

## Ровно ли растёт панель

Теперь главное для когортного анализа: как меняется число клиентов по месяцам. Если выгрузка
формировалась одинаково все 17 месяцев, рост должен быть плавным.

In [ ]:
months = con.execute("""
SELECT m AS месяц, count(*) AS клиентов,
       round(100.0*count(*) FILTER (WHERE n_prod = 0)/count(*), 1) AS без_продуктов_проц
FROM p GROUP BY 1 ORDER BY 1""").fetchdf()
months['прирост'] = months['клиентов'].diff()
months

А вот и разрыв. Июнь 2015 - 632 110 клиентов, июль - 829 817, то есть за один месяц плюс
почти 198 тысяч. Одновременно доля клиентов без единого продукта прыгает с 2,5% до 24,6%
и дальше держится около 25%.

Два скачка сразу в одном месяце - это не поведение клиентов, это смена правил выгрузки.
Проверим: если в июле пришли действительно новые клиенты, у них должна быть свежая `fecha_alta`.

In [ ]:
con.execute("""
WITH july AS (SELECT ncodpers FROM fs WHERE m1 = DATE '2015-07-01')
SELECT year(p.cohort) AS год_подключения, count(*) AS клиентов
FROM july JOIN p USING (ncodpers)
WHERE p.m = DATE '2015-07-01'
GROUP BY 1 ORDER BY 2 DESC LIMIT 8""").fetchdf()

Нет, они не новые: 26,7 тыс. подключились в 2014-м, 19,1 тыс. в 2013-м, 15,4 тыс. в 2012-м и так
далее вглубь до 2001 года. Значит в июле 2015 в выгрузку добавили давно существующих клиентов,
которых раньше просто не отдавали.

Это важнее, чем кажется. Посмотрим, кто именно держит нулевые наборы продуктов.

In [ ]:
con.execute("""
SELECT CASE WHEN fs.m1 <= DATE '2015-06-01' THEN '1. были до июля 2015'
            WHEN fs.m1  = DATE '2015-07-01' THEN '2. добавлены в июле 2015'
            ELSE '3. пришли после июля' END AS группа,
       count(DISTINCT p.ncodpers) AS клиентов,
       round(100.0*count(*) FILTER (WHERE p.n_prod = 0)/count(*), 1) AS строк_без_продуктов_проц
FROM p JOIN fs USING (ncodpers) GROUP BY 1 ORDER BY 1""").fetchdf()

Вот и объяснение: у клиентов, добавленных в июле, в 91,1% строк нет ни одного продукта, тогда
как у «старых» - только в 3,7%. То есть июльское расширение - это в основном клиенты без
продуктов: формально они в базе, фактически банку не приносят ничего.

Практический вывод: считать по всей панели «у нас 25% клиентов без продуктов» нельзя - это
артефакт выгрузки, а не свойство клиентской базы.

## Мусор в полях

Вернёмся к колонкам, которые приехали строками. Посмотрим на `age` и `antiguedad` и заодно
проверим, не один ли это набор испорченных строк.

In [ ]:
con.execute("""
SELECT count(*) FILTER (WHERE TRY_CAST(trim(age) AS INT) IS NULL) AS age_нечисло,
       min(TRY_CAST(trim(age) AS INT)) AS age_min,
       max(TRY_CAST(trim(age) AS INT)) AS age_max,
       count(*) FILTER (WHERE TRY_CAST(trim(antiguedad) AS INT) IS NULL) AS antiguedad_нечисло,
       min(TRY_CAST(trim(antiguedad) AS INT)) AS antiguedad_min,
       count(*) FILTER (WHERE TRY_CAST(trim(antiguedad) AS INT) = -999999) AS antiguedad_999999,
       count(*) FILTER (WHERE fecha_alta IS NULL) AS fecha_alta_пусто,
       count(*) FILTER (WHERE TRY_CAST(trim(age) AS INT) IS NULL
                          AND TRY_CAST(trim(antiguedad) AS INT) IS NULL
                          AND fecha_alta IS NULL) AS все_три_сразу
FROM t""").fetchdf()

Интересно: нечисловых `age` ровно 27 734, нечисловых `antiguedad` тоже 27 734, пустых
`fecha_alta` снова 27 734, и все три условия выполняются одновременно на тех же строках.
Значит это не три независимых дефекта, а один блок битых строк, где клиентская часть записи
просто не заполнилась.

Отдельно: `antiguedad` содержит служебное -999999 (38 строк), а максимальный `age` - 164 года,
что физически невозможно. В когортах эти значения не участвуют, но при разрезах по возрасту и
стажу их надо отсекать.

Посмотрим, когда жил этот блок.

In [ ]:
con.execute("""
SELECT date_trunc('month', fecha_dato) AS месяц, count(*) AS битых_строк,
       count(DISTINCT ncodpers) AS клиентов
FROM t WHERE fecha_alta IS NULL GROUP BY 1 ORDER BY 1""").fetchdf()

Блок живёт только в первой половине 2015 года и затухает: январь 6953 строки, февраль 5940,
дальше по убывающей до 1861 в июне, а с июля - ни одной. Похоже, дефект выгрузки, который
к июлю починили. Всего затронуто 7340 клиентов.

## Пропуски в полях для разрезов

In [ ]:
con.execute("""
SELECT round(100.0*count(*) FILTER (WHERE renta IS NULL)/count(*), 2) AS renta,
       round(100.0*count(*) FILTER (WHERE canal IS NULL)/count(*), 2) AS canal_entrada,
       round(100.0*count(*) FILTER (WHERE segmento IS NULL)/count(*), 2) AS segmento,
       round(100.0*count(*) FILTER (WHERE rel IS NULL)/count(*), 2) AS tiprel_1mes
FROM p""").fetchdf()

`renta` (доход домохозяйства) пропущена в 20,5% строк - это много, и заполнять её медианой
по стране смысла нет. Будем использовать её только как дополнительный разрез, а не как основу
выводов. Канал, сегмент и тип связи пропущены примерно в 1% строк - с этим можно работать.

## Дырки в истории клиента

Для оттока это критично: если клиент пропал на один месяц и вернулся, это не уход. Посчитаем,
у скольких клиентов число месяцев в панели меньше, чем размах между первым и последним месяцем.

In [ ]:
con.execute("""
WITH sp AS (SELECT ncodpers, count(*) AS месяцев,
                   datediff('month', min(m), max(m)) + 1 AS окно
            FROM p GROUP BY 1)
SELECT count(*) AS клиентов,
       count(*) FILTER (WHERE месяцев < окно) AS с_дырками,
       round(100.0*count(*) FILTER (WHERE месяцев < окно)/count(*), 2) AS доля_проц
FROM sp""").fetchdf()

8018 клиентов (0,84%) имеют разрывы. Доля небольшая, но правило всё равно нужно: прежде чем
признать клиента ушедшим, проверяем, не появляется ли он в следующих месяцах.

## Кого нельзя записывать в отток

В данных есть служебные признаки, которые означают не уход по своей воле, а другое событие.

In [ ]:
con.execute("""
SELECT count(DISTINCT ncodpers) FILTER (WHERE trim(indfall) = 'S') AS умерли,
       count(DISTINCT ncodpers) FILTER (WHERE trim(indrel_1mes) IN ('3','3.0','4','4.0')) AS бывшие,
       count(DISTINCT ncodpers) FILTER (WHERE trim(indrel) = '99') AS ушли_внутри_месяца,
       count(DISTINCT ncodpers) FILTER (WHERE ult_fec_cli_1t IS NOT NULL) AS есть_дата_ухода
FROM t""").fetchdf()

2731 умерший клиент, 4484 «бывших» по `indrel_1mes`, 19 889 с заполненной датой ухода
`ult_fec_cli_1t`. Смерть клиента - не отток в продуктовом смысле, её надо считать отдельно,
иначе получим управленческий вывод там, где ничего нельзя сделать.

## Какие когорты вообще можно считать

Теперь соберём всё вместе. Когорту определяем по месяцу `fecha_alta`. Но панель начинается в
январе 2015, а в июле её расширили, поэтому не у каждой когорты видно начало жизни. Проверим,
какая доля когорты появляется в панели в свой же месяц подключения.

In [ ]:
con.execute("""
SELECT cohort AS когорта, count(*) AS клиентов,
       count(*) FILTER (WHERE m1 = cohort) AS видны_с_первого_месяца,
       round(100.0*count(*) FILTER (WHERE m1 = cohort)/count(*), 1) AS доля_проц
FROM fs WHERE cohort >= DATE '2015-01-01'
GROUP BY 1 ORDER BY 1""").fetchdf()

Картина однозначная: когорты января - июня 2015 видны с первого месяца только на 54-59%,
а начиная с июля - на 99,2-99,9%. Причина та же: до июля выгрузка отдавала не всех.

Отсюда рабочее окно: **когорты с июля 2015 по апрель 2016**. Апрель, а не май, потому что
май 2016 - последний снимок, и для него отток не определён (правое цензурирование).

И ещё одно правило: в когорту берём только тех, у кого первый месяц в панели совпал с месяцем
подключения. Иначе клиенты с ранней `fecha_alta` наполняют свои когорты задним числом, и
матрица удержания выдаёт значения выше 100% - я на этом сначала и споткнулась.

Посмотрим, сколько клиентов остаётся при таком условии.

In [ ]:
con.execute("""
SELECT count(DISTINCT ncodpers) FILTER (WHERE cohort < DATE '2015-01-01') AS усечены_слева,
       count(DISTINCT ncodpers) FILTER (WHERE cohort >= DATE '2015-01-01') AS внутри_окна,
       count(DISTINCT ncodpers) FILTER (WHERE cohort IS NULL) AS без_даты_подключения,
       count(DISTINCT ncodpers) FILTER (WHERE m1 = cohort
             AND cohort BETWEEN DATE '2015-07-01' AND DATE '2016-04-01') AS годятся_для_когорт
FROM fs""").fetchdf()

## Итоги

Что мы узнали и как это меняет расчёт оттока:

1. **Ключ чистый.** 13 647 309 строк, 956 645 клиентов, пара (`ncodpers`, `fecha_dato`) без дублей.
2. **В июле 2015 выгрузку расширили** на ~198 тыс. клиентов с давней `fecha_alta`, и у них в
   91% строк нет ни одного продукта. Поэтому «25% клиентов без продуктов» по всей панели -
   артефакт, а не факт о базе.
3. **Рабочее окно когорт - июль 2015 - апрель 2016**, членство когорты только для тех, чей первый
   месяц в панели совпал с месяцем подключения. Годятся 127 034 клиента.
4. **Один блок битых строк** (27 734 строки, 7340 клиентов): пустая `fecha_alta` и нечисловые
   `age` и `antiguedad` живут на одних и тех же строках и только в первой половине 2015 года.
5. **`renta` пропущена в 20,5%** - только вспомогательный разрез. Канал, сегмент и тип связи - около 1%.
6. **0,84% клиентов имеют дырки в истории** - пропущенный месяц не считаем оттоком.
7. **2731 умерший и 4484 «бывших» клиента** - отдельная категория, не отток.

Дальше - `02_cohorts_churn.ipynb`: когортные матрицы удержания, отток по продуктам с отсечением
миганий флага и проверка гипотез.